# Spark-X2.6-1.7B — on-device agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Spark-X2.6-1.7B-on-device-agent.ipynb)

[`XHToken/Spark-X2.6-1.7B`](https://huggingface.co/XHToken/Spark-X2.6-1.7B) is SparkLLM's compact on-device agent (~1.71B, Apache 2.0, English + Chinese). Same `spark2_5` architecture as Spark-X2.5-1.7B; the gains are from post-training. The chat template takes `enable_thinking` plus `reasoning_effort` (`low` / `high`) and a `tools` list in OpenAI function form.

It is the smaller sibling of [`Spark-X2.6-4B`](Spark-X2.6-4B-on-device-agent.ipynb). SparkLLM's [announcement](https://x.com/SparkLLM/status/2115100695775056057) reports **SWE Verified 59.8 vs 62.8** for the 4B and **τ²-bench 85.3 vs 86.9**, both at high thinking — less than half the parameters, close on those two scores. Float16 weights are about **3.5 GB**, so a free T4 does **not** need 4-bit (the 4B is ~8.2 GB in float16 and may fall back to NF4).

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Spark-X2.6-1.7B-on-device-agent.md) for when this 1.7B is a good fit and when it is not.

**Licence:** Apache 2.0. Weights are public — `HF_TOKEN` is optional.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The card pins `transformers==4.57.1` and loads with `trust_remote_code=True` (`modeling_spark.py`). `torch_dtype="auto"` is bfloat16; a T4 has no native bfloat16, so this notebook loads **float16**. The shipped modeling code uses eager attention only, so FlashAttention 2 is not on this path. The card's `max_new_tokens=131072` does not fit a free T4; every demo stays in the low hundreds / low thousands.

**Links:** [model card](https://huggingface.co/XHToken/Spark-X2.6-1.7B) · [announcement](https://x.com/SparkLLM/status/2115100695775056057) · [4B notebook](Spark-X2.6-4B-on-device-agent.ipynb) · [GGUF](https://huggingface.co/XHToken/Spark-X2.6-1.7B-GGUF) (`Q4_K_M` ~1.8 GB)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so a later transformers pin can take effect without a restart. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# modeling_spark.py implements eager attention only. Keep SDPA on the
# mem-efficient and math kernels in case a later generate path touches them.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16 (~3.5 GB weights, no 4-bit).")


## 2. Install the pinned transformers

The [model card](https://huggingface.co/XHToken/Spark-X2.6-1.7B#transformers) requires `pip install transformers==4.57.1`. Colab ships a newer copy. This cell downgrades it, then drops any already-imported `transformers` modules so **Runtime > Run all** can pick up 4.57.1 without *Restart session*. If the version assert still fails, restart once and continue from the GPU cell.

Do not reinstall torch. `accelerate` is for `device_map="auto"`. **bitsandbytes is not installed** — 1.7B in float16 fits a T4.


In [ ]:
%pip install -q "transformers==4.57.1" "accelerate>=1.1.0"

import sys

# Drop a pre-imported Colab transformers so 4.57.1 is visible without Restart session.
for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__) == Version("4.57.1"), (
    f"transformers {transformers.__version__} is loaded; Spark-X2.6-1.7B's card pins 4.57.1. "
    "Runtime > Restart session, then rerun from the GPU check cell."
)
print("transformers", transformers.__version__)


## 3. Load Spark-X2.6-1.7B

`AutoModelForCausalLM` and `AutoTokenizer`, as on the [model card](https://huggingface.co/XHToken/Spark-X2.6-1.7B#transformers). `trust_remote_code=True` is required: the checkpoint is `Spark2_5ForCausalLM`, not a stock transformers class. The first run downloads about 3.5 GB.

The card's generate snippet uses temperature 1.0, top_p 0.95, top_k 0. The saved `generation_config.json` stores `top_k: -1` (transformers rejects that, so the helper always passes `top_k=0`) and `repetition_penalty=1.0`. The Jinja / SGLang default is thinking **on** with `reasoning_effort="high"`. The think demo below uses **`low`** so a T4 token budget can close `</think>` and still print a final answer (the card: high→low is 37.3% shorter on this 1.7B). Coding and tool demos turn thinking off so tokens go to the function or the call.


In [ ]:
import gc
import re

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "XHToken/Spark-X2.6-1.7B"


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    device_map="auto",
).eval()
model.generation_config.top_k = 0

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", f"{n_params / 1e9:.2f}B")
print("dtype:", next(model.parameters()).dtype)
print_vram("after load")


THINK_CLOSE = "</think>"


def split_think(text, *, thinking_enabled=False):
    # Card chat template / SGLang --reasoning-parser qwen3: enable_thinking=True
    # prepends <think> on the generation prompt. New tokens are the think
    # channel until </think>, then the response channel. An unclosed think
    # block is still think, not the answer. <think>/</think> are added tokens
    # with special=false, so skip_special_tokens keeps them in the decode.
    if THINK_CLOSE in text:
        think, answer = text.split(THINK_CLOSE, 1)
        return think.replace("<think>", "").strip(), answer.strip()
    if thinking_enabled or text.lstrip().startswith("<think>"):
        return text.replace("<think>", "").strip(), ""
    return "", text.strip()


def generate_ids(input_ids, max_new_tokens, temperature):
    # Card Transformers snippet + generation_config.json (top_k=-1 → 0).
    with torch.inference_mode():
        return model.generate(
            input_ids=input_ids,
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            top_k=0,
            repetition_penalty=1.0,
        )


def decode_new(generated, prompt_len):
    return tokenizer.decode(generated[0, prompt_len:], skip_special_tokens=True)


def chat(
    messages,
    *,
    tools=None,
    enable_thinking=False,
    reasoning_effort="high",
    max_new_tokens=256,
    temperature=1.0,
):
    template = {
        "tokenize": True,
        "add_generation_prompt": True,
        "return_dict": True,
        "return_tensors": "pt",
        "enable_thinking": enable_thinking,
    }
    if enable_thinking:
        template["reasoning_effort"] = reasoning_effort
    if tools is not None:
        template["tools"] = tools
    inputs = tokenizer.apply_chat_template(messages, **template)
    device = next(model.parameters()).device
    input_ids = inputs["input_ids"].to(device)
    generated = generate_ids(input_ids, max_new_tokens, temperature)
    text = decode_new(generated, input_ids.shape[-1])
    if enable_thinking and THINK_CLOSE not in text:
        # High thinking on this 1.7B can fill the T4 cap before </think>.
        # Drop a trailing EOS if present, append the close token, then spend a
        # short budget on the response channel.
        eos_id = tokenizer.eos_token_id
        seeded = generated
        if eos_id is not None and int(seeded[0, -1]) == eos_id:
            seeded = seeded[:, :-1]
        close_ids = tokenizer.encode(THINK_CLOSE, add_special_tokens=False)
        seeded = torch.cat([seeded, seeded.new_tensor([close_ids])], dim=1)
        generated = generate_ids(seeded, min(384, max_new_tokens), temperature)
        text = decode_new(generated, input_ids.shape[-1])
    print_vram()
    return text


## 4. Think-then-answer chat

`enable_thinking=True` opens a `<think>` channel (the card's Jinja template; SGLang's `--reasoning-parser qwen3`). The card / SGLang default is `reasoning_effort="high"`; this demo uses **`low`**, which the card reports as 37.3% shorter on the 1.7B, so a T4 budget can close `</think>` and still write the response channel. Sampling matches the Transformers snippet and `generation_config.json` (`do_sample=True`, `temperature=1.0`, `top_p=0.95`, `top_k=0`, `repetition_penalty=1.0`). `max_new_tokens=1024` is far below the card's 131072. If the think channel still does not close, the helper appends `</think>` and generates the answer. The printed block splits think from answer.


In [ ]:
think_messages = [{
    "role": "user",
    "content": (
        "A free T4 has 16 GB of VRAM. Spark-X2.6-1.7B has about 1.71B parameters; "
        "the 4B sibling has about 4.11B. About how many gigabytes is the 1.7B in float16, "
        "does it fit this GPU without 4-bit, and how does that compare with the 4B in float16? "
        "Think briefly in the think channel, then close it and answer in two complete sentences."
    ),
}]
think_text = chat(
    think_messages,
    enable_thinking=True,
    reasoning_effort="low",
    max_new_tokens=1024,
    temperature=1.0,
)
think, answer = split_think(think_text, thinking_enabled=True)
print("----- think -----")
print(think or "(empty think channel)")
print("----- answer -----")
print(answer or "(no response channel)")


## 5. Coding task

Ask for one small parser, then run the generated function in a later cell against a fixed test file. The generate cell only writes source; it does not execute it. Thinking stays off so tokens go to the function. Temperature 0.4 is a demo choice (the card's sample uses 1.0).


In [ ]:
CODE_PROMPT = '''Write one Python function named parse_kv and nothing else.

Signature:
    def parse_kv(text: str) -> dict[str, str]:

Rules:
- each non-empty line is key=value
- skip blank lines and lines whose first non-space character is #
- strip whitespace around the key and the value
- if a key appears twice, the last assignment wins
- do not import anything; do not add a main block

Put the full function in one python code fence.'''

code_messages = [{"role": "user", "content": CODE_PROMPT}]
code_text = chat(code_messages, enable_thinking=False, max_new_tokens=400, temperature=0.4)
_, code_body = split_think(code_text)
fences = re.findall(r"```(?:python)?\n(.*?)```", code_body, flags=re.DOTALL)
SOLUTION_SRC = (fences[-1] if fences else code_body).strip() + "\n"
print(SOLUTION_SRC)


The next cell writes that source to a temp file, rejects a short list of dangerous imports, and runs `python -s` against four assertions. `-s` skips user site-packages and still puts the script directory on `sys.path`, so `from solution import parse_kv` works.


In [ ]:
import ast
import subprocess
import tempfile
from pathlib import Path

TESTS = r'''
from solution import parse_kv

assert parse_kv("host=localhost\nport=8080") == {"host": "localhost", "port": "8080"}
assert parse_kv("# comment\nname=a\nname=b\n") == {"name": "b"}
assert parse_kv("  token = abc  \n\n") == {"token": "abc"}
assert parse_kv("") == {}
print("all tests passed")
'''

BANNED = {"os", "subprocess", "socket", "pathlib", "shutil", "sys", "ctypes", "importlib", "requests", "urllib", "http", "pickle"}


def check_source(source):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                root = alias.name.split(".")[0]
                if root in BANNED:
                    raise SystemExit(f"blocked import: {alias.name}")
        elif isinstance(node, ast.ImportFrom):
            root = (node.module or "").split(".")[0]
            if root in BANNED:
                raise SystemExit(f"blocked import: {node.module}")
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {"eval", "exec", "compile", "__import__"}:
            raise SystemExit(f"blocked call: {node.func.id}")


print("generated source\n")
print(SOLUTION_SRC)
check_source(SOLUTION_SRC)

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    (tmp_path / "solution.py").write_text(SOLUTION_SRC)
    (tmp_path / "test_parse_kv.py").write_text(TESTS)
    proc = subprocess.run(
        ["python", "-s", "test_parse_kv.py"],
        cwd=tmp,
        capture_output=True,
        text=True,
        timeout=8,
    )
print(proc.stdout)
if proc.returncode:
    print(proc.stderr)
    raise SystemExit(f"tests failed with exit {proc.returncode}")


## 6. Tool-use loop

The chat template renders `tools` under a `<tools>` block and expects calls as `<tool_call>name<arg_key>…</arg_key><arg_value>…</arg_value></tool_call>`. Thinking stays off so the first generate spends tokens on the call, not a trace. Stubs stand in for weather and local time; `get_local_time` returns ISO, a 24-hour clock, and a `time_of_day` label so the final answer can match the tool instead of guessing morning/afternoon from the prompt. Swap the stubs for real APIs if you want.


In [ ]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name, e.g. Singapore"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_local_time",
            "description": (
                "Get the current local time in an IANA timezone, including a "
                "24-hour clock (HH:MM) and a time-of-day label "
                "(morning, afternoon, evening, or night)."
            ),
            "parameters": {
                "type": "object",
                "properties": {"timezone": {"type": "string", "description": "IANA timezone, e.g. Asia/Singapore"}},
                "required": ["timezone"],
            },
        },
    },
]


def get_weather(city: str) -> dict:
    return {"city": city, "celsius": 31, "condition": "partly cloudy"}


def time_of_day_label(hour: int) -> str:
    if 5 <= hour < 12:
        return "morning"
    if 12 <= hour < 17:
        return "afternoon"
    if 17 <= hour < 21:
        return "evening"
    return "night"


def get_local_time(timezone: str) -> dict:
    now = datetime.now(ZoneInfo(timezone))
    return {
        "timezone": timezone,
        "iso": now.isoformat(timespec="seconds"),
        "local_24h": now.strftime("%H:%M"),
        "time_of_day": time_of_day_label(now.hour),
    }


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_local_time": get_local_time}


def parse_tool_calls(text):
    calls = []
    for block in re.finditer(r"<tool_call>(.*?)</tool_call>", text, flags=re.DOTALL):
        body = block.group(1)
        if "<arg_key>" in body:
            name, rest = body.split("<arg_key>", 1)
            args = {}
            for match in re.finditer(
                r"<arg_key>(.*?)</arg_key><arg_value>(.*?)</arg_value>",
                "<arg_key>" + rest,
                flags=re.DOTALL,
            ):
                key, value = match.group(1), match.group(2)
                try:
                    args[key] = json.loads(value)
                except json.JSONDecodeError:
                    args[key] = value
        else:
            name, args = body, {}
        calls.append({"name": name.strip(), "arguments": args})
    return calls


tool_messages = [{
    "role": "user",
    "content": (
        "I am in Singapore right now. What is the weather, and what time is it there? "
        "Use the tools, then give a two-sentence recommendation that states the "
        "returned local time (24-hour clock) and the tool's time_of_day label. "
        "Do not assume morning, afternoon, evening, or night from this prompt."
    ),
}]
first = chat(tool_messages, tools=TOOLS, enable_thinking=False, max_new_tokens=256, temperature=0.4)
think, rest = split_think(first)
print("----- first generate -----")
print(first)

calls = parse_tool_calls(rest)
print("parsed calls:", calls)

if not calls:
    print("No <tool_call> block. The model answered in text, or the call was malformed.")
else:
    content = re.sub(r"<tool_call>.*?</tool_call>", "", rest, flags=re.DOTALL).strip()
    assistant = {
        "role": "assistant",
        "content": content,
        "tool_calls": [
            {"function": {"name": call["name"], "arguments": call["arguments"]}} for call in calls
        ],
    }
    if think:
        assistant["reasoning_content"] = think
    follow_messages = [*tool_messages, assistant]
    for call in calls:
        fn = TOOL_FUNCTIONS.get(call["name"])
        if fn is None:
            payload = {"error": f"unknown tool: {call['name']}"}
        else:
            try:
                payload = fn(**call["arguments"])
            except TypeError as exc:
                payload = {"error": str(exc)}
        print("tool", call["name"], payload)
        follow_messages.append({"role": "tool", "content": json.dumps(payload, separators=(",", ":"))})
    final = chat(follow_messages, tools=TOOLS, enable_thinking=False, max_new_tokens=220, temperature=0.4)
    _, final_answer = split_think(final)
    print("----- final -----")
    print(final_answer or final)


## 7. Caveats and cleanup

- **Custom modeling code.** The checkpoint is `Spark2_5ForCausalLM`. Loading it runs `modeling_spark.py` and `configuration_spark.py` from the Hub (`trust_remote_code=True`). Read those files before you use this in anything that is not a notebook.
- **Pinned transformers.** The card requires `transformers==4.57.1`. Colab's preinstalled copy is newer. The install cell downgrades it and drops cached modules so Run all usually works; if the version assert fails, *Restart session* once.
- **Float16 on a T4, no 4-bit.** The saved dtype is bfloat16. A T4 has no native bfloat16, so this notebook uses float16 (~3.5 GB of weights). Unlike the 4B sibling (~8.2 GB fp16), 4-bit is not needed on 16 GB. Newer GPUs (L4, A100) can load bfloat16, the card default.
- **Modest generation length.** The card's snippet uses `max_new_tokens=131072` and advertises a 1,048,576-token context. Neither fits a 16 GB T4. Demos here stay in the low hundreds / ~1k new tokens. The think demo uses `reasoning_effort="low"` (card: 37.3% shorter than high on this 1.7B) with a 1024-token cap; if `</think>` never appears, the helper closes the think channel and generates the answer so the printed block is not truncated reasoning.
- **Optional GGUF.** [`XHToken/Spark-X2.6-1.7B-GGUF`](https://huggingface.co/XHToken/Spark-X2.6-1.7B-GGUF) has `Spark-X2.6-1.7B-Q4_K_M.gguf` (~1.8 GB). Native `spark2_5` support needs [llama.cpp](https://github.com/ggml-org/llama.cpp/) `b10828` or later. Prefer the transformers float16 path above if it fits.

The next cell drops the model so you can load something else on the same T4.


In [ ]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
